# California Residential Home Price Prediction
# School District Spatial Join

This notebook adds school district information as a feature by performing a spatial join between property coordinates and California Unified School District boundaries. This replaces the incomplete HighSchoolDistrict column, which was missing for about 25 percent of records, with a complete and geographically accurate feature.

In [27]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from sklearn.preprocessing import LabelEncoder

## Section 1 - Setup and Data Loading

In [28]:
df_model = pd.read_csv('data/cleaned_data.csv')
print('Shape:', df_model.shape)

school_districts = gpd.read_file('data/DistrictAreas2526_-284845464123469011.geojson')
print('School districts shape:', school_districts.shape)
print(school_districts.columns.tolist())

Shape: (397461, 23)
School districts shape: (936, 51)
['OBJECTID', 'Year', 'FedID', 'CDCode', 'CDSCode', 'CountyName', 'DistrictName', 'DistrictType', 'GradeLow', 'GradeHigh', 'GradeLowCensus', 'GradeHighCensus', 'AssistStatus', 'UpdateNotes', 'EnrollTotal', 'EnrollCharter', 'EnrollNonCharter', 'AAcount', 'AApct', 'AIcount', 'AIpct', 'AScount', 'ASpct', 'FIcount', 'FIpct', 'HIcount', 'HIpct', 'PIcount', 'PIpct', 'WHcount', 'WHpct', 'MRcount', 'MRpct', 'NRcount', 'NRpct', 'ELcount', 'ELpct', 'FOScount', 'FOSpct', 'HOMcount', 'HOMpct', 'MIGcount', 'MIGpct', 'SWDcount', 'SWDpct', 'SEDcount', 'SEDpct', 'DistrctAreaSqMi', 'LocaleCode', 'LocaleDesc', 'geometry']


## Section 2 - Filtering to Unified School Districts

Per instructions, only Unified school districts are used for this join, since they cover the full grade range and provide a single, consistent geographic boundary per area, avoiding the overlapping elementary and high school district boundaries.

In [29]:
unified_districts = school_districts[school_districts['DistrictType'] == 'Unified'].copy()
print('Unified districts shape:', unified_districts.shape)

Unified districts shape: (345, 51)


## Section 3 - Converting Coordinates to Geographic Points

Each property's Latitude and Longitude are converted into a Point geometry so they can be spatially compared against the school district polygons.

In [30]:
geometry = [Point(xy) for xy in zip(df_model['Longitude'], df_model['Latitude'])]
properties_gdf = gpd.GeoDataFrame(df_model, geometry=geometry, crs='EPSG:4326')
properties_gdf = properties_gdf.to_crs(unified_districts.crs)

print('Properties GeoDataFrame shape:', properties_gdf.shape)

Properties GeoDataFrame shape: (397461, 24)


## Section 4 - Spatial Join

A spatial join determines which Unified School District polygon contains each property's coordinates, using a point in polygon operation.

In [31]:
joined = gpd.sjoin(properties_gdf, unified_districts[['DistrictName', 'geometry']], how='left', predicate='within')

print('Joined shape:', joined.shape)
print('Missing DistrictName:', joined['DistrictName'].isnull().sum())

Joined shape: (397461, 26)
Missing DistrictName: 100030


In [32]:
joined['DistrictName'] = joined['DistrictName'].fillna('Not Unified')
print(joined['DistrictName'].value_counts().head(10))

DistrictName
Not Unified             100030
Los Angeles Unified      37749
San Diego Unified        10511
Capistrano Unified        7034
Desert Sands Unified      7002
Palm Springs Unified      6000
Oakland Unified           5273
Corona-Norco Unified      5098
Hemet Unified             4913
Long Beach Unified        4769
Name: count, dtype: int64


## Section 5 - Finalizing the Enriched Dataset

DistrictName is label encoded for model compatibility, replacing the incomplete HighSchoolDistrict column with a complete, spatially accurate feature covering all properties.

In [33]:
joined = joined.drop(columns=['index_right'])

le = LabelEncoder()
joined['DistrictName'] = le.fit_transform(joined['DistrictName'])

joined = joined.drop(columns=['HighSchoolDistrict', 'geometry'])

print('Final shape:', joined.shape)
print('Columns:', joined.columns.tolist())

Final shape: (397461, 23)
Columns: ['ViewYN', 'PoolPrivateYN', 'ClosePrice', 'Latitude', 'Longitude', 'LivingArea', 'CountyOrParish', 'AttachedGarageYN', 'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'Stories', 'Levels', 'GarageSpaces', 'PostalCode', 'AssociationFee', 'LotSizeSquareFeet', 'PropertyAge', 'BedBathRatio', 'CloseMonth', 'CloseYear', 'ListingDuration', 'DistrictName']


## Section 6 - Summary

The spatial join successfully assigned a Unified School District to about 75 percent of properties, with the remaining 25 percent falling in areas covered by separate Elementary or High school districts, labeled as Not Unified. This provides a complete geographic feature, replacing the previous HighSchoolDistrict column which was 25 percent missing.

In [34]:
joined.to_csv('data/cleaned_data_with_district.csv', index=False)
print('Saved.')

Saved.
